Multiple versions of a recommender were built with Association Rule Learning, Collaborative Filtering and Content Based Filtering used.

# Summary and Analysis

# Scraping
Data was scraped from MyAnimeList with additional fake_user_credentials used to mask speed up requests sent.

No restrictions were found in the robots.txt regarding the pages used.

# Preprocessing
Strings were parsed into their correct formats.

.pkl file formatwere used instead in scenarios requiring array data.

Some other features were also generated to help filtering such a name's vowel percentage and consonant amount to determine if personnel is Japanese.

# Association Rule Learning
ARules is a powerful tool for market basket and frequent item set analysis.

This is the first version encountered when researching for casting recommender systems.

However it is extremely time consuming and resource intensive in generating rules thus a different approach is explored.

If the notebook 2. Arules.ipnyb is not loaded, it's most likely that even the team ran out PC resources to run the file. In particular, a Blue Screen and NVIDIA memory warnings.

In addition, it's a frequency based tool. This not meaningful in providing viewer and audience perception of the VA's role.

For example, a small time VA might be able to take multiple small anime projects while a full time main cast will be more restricted if they have the main role.

# Collab
Collaborative filtering is the approach taken.

It improves upon ARules in terms of resource performance and time. However the precision it generates is rather poor.

This is because of issues with data sparsity.

Casting is intentionally sparse thus trying to generate values/impute with Matrix Factorization may just introduce noise.

Even after more extensive preprocessing such as stricter filters to reduce sparsity and nationality checks, the results can only improve until a threshold.

# Content Based

Content based was the best in terms of precision.

It's able to take previous projects a VA has taken, generate similarity values per feature, and reccommend the best match. 

Compared to the rest it's light, since it requires a separate DB for items (projects) and another DB for user projects taken.

The projects taken is will select necessary items and average their values to find the average features of projects taken.

# Evaluation
Precision was the main metric used in evaluating train and test sets.

This is because the interest lies in measuring the prediction.

Is it a False Positive or True Positive.

It's natural for job applications/role castings to be turned down, hence providing a reccomender system that guarantees a metric that it's prediction is correct is what's meaningful.

# --Revisions--

## ARules

Association Rules looks at the history of all anime projects and treats the casting members as a list of transactions.

By evaluating enough of the the project casts, it is able to generate frequent itemsets and rules for us to analyze.

To filter these down to a managable and relevant size, our hyperparamters are Support, Confidence, and Zmin.

![](Scores/ARules_Confidence.png)

Association Rules main driver for improving precision is confidence. It's computation (XUY / X) is equivalent to computing for precision (TP/TP+FP) hence the directly correlated relationship

![](Scores/ARules_Support.png)

Since confidence has the highest weight of all parameters, it can tend to weaken the weights of Support.

## Collaborative
Collaborative filtering uses the standaridezed performance of scores items (projects) and users (actors).

A pivot table was then generated with SVD imputation being an effective tool in improving user similarity.

From this a user-user similarity matrix was generated to recommend actors that had similar (positive), neutral, or negative performances.

#### Basic Sparse Model
![](Scores/UserUserPrecision.csv.png)

The most noticeable trend is that as the prediction pool increases, the chances for errors also increases as more guesses also allow more chances of getting it wrong.

Another observation is that the train scores are higher than the test scores. Too high the difference and there is a chance of a overfit.

Lastly is the trend on the similar actor refernces used. The more actor references are used, the higher the precision becomes but it then flatlines thus possibly indicating that the first at the elbow were already sufficient enough as references. The subsequent are just too irrelevant to make an impact.


#### SVD Sparse Model
![](Scores/UserUserSVDPrecision150.csv.png)

SVD is able to improve the precision of the first few references used. Note that it didn't signifcantly alter values, but rather it shifted the similarities.

Instead of peak predictions being achieved at the elbow, it is now shifted to the first few.

However, it still has difficulty on the Test set as these were the masked values, hence the imputation generated may have treated these equally as a sparse/missing value.

#### SVD Less Sparse Model (JP Subset)
![](Scores/UserUserSVDPrecision150_JP.csv.png)

If sparsity was addressed earlier in preprocessing drops, scores could further significantly improve.

The Japanese only voice actors had a sparsity of 11% thus SVD is seen to not just shift the peak like before, but also inreasese its maximum precision potential.

The downward trend is now also more severe thereby emphasizing the dissimalirity / non-synergy of subsequent actors.

## Content Based

Content Based builds an item matrix using the projects' one hot encoded features of genre, theme, etc.

To reccomend similar projects to a Voice Actor, they must present a portion of their work history and the system will reccomend those projects with closest average features to those they already starred in. Projects in the presented history is omitted from reccomendations genrated.


![](Scores/ContenBased.png)

This time only the Top 10 recommendations was used, since more predictions just dilutes precision as seen before. 

Compared to collaborative, content based is less trend determinable.

There should be a specific range of references used to generate the best recommnedations.

Too few and it might be hyper fixated and too many will produce noise.



### Scaling Weights
Compared to the other approaches, this one was aims to recommend projects instead.

It could easily be altered to recommend similar actors as well by joining the anime featuress with the actor's work history, thereby resulting in an Actor-Anime Features records.

However the downside is already aparent as frequency and features are once again not indicative of quality.

To address this, an additional scaling of weights was applied to the Top 10 most similar (highest mean value of work references) features.

![](Images/ScaledWeights.png)

However the most apparent issue should already be stated.

The features are too many. Much of the similar features would start becoming noise if less relevant features like Studio, Licensor, etc. are included. These are technical and contract based items that many VA are not restricted to since they're not exclusive employees of specific studios. They're free to take on whichever project is preffered unlike staff.

# Conclusion

The reccomender systems created aim to serve different purposes and have their own caveats.


## ARules
For producers aiming to find cast members with strong history.

It can easily form a team by specifiying minimum projects taken, precision/confidence of hiring success, and the desired number of team members.

It's downside is that can't work with information such as relevance to the genre of the project being developed or the VA team's performance.

## Colab
For producers wanting to hire manually through synergizing connections.

Synergy refers to shared projects where their performances were roughly equally rated. 

It recommends co-cast that fit the criteria.

The drawbacks are it's heavily filtered already thus any VA recommended must be one of professional standard with at least 100 projects taken with notable community likes for that role.


## Content Based
For voice actors wanting to look for their next project.

This recommends similar projects based on their feature similarity to those already starred in.

It's advised to provide a strong history as well to be able to properly weigh the most important features of a project the VA is willing to take.

The caveat to this approach, is that once again performance rating is lost thus recommendation is on the most frequent features instead.